In [33]:
import pandas as pd
import json
import re
import time
from tqdm.auto import tqdm
from google import genai
from google.genai import types

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
# SECURITY: Do not paste your key directly in the script if sharing code.
# Use a file or environment variable.
api_key = "REPLACE_WITH_YOUR_OWN_KEY" 

# Initialize Google Gen AI Client
client = genai.Client(api_key=api_key)

# CHANGED: 'gemini-3-pro-preview' has NO free tier. 
# Switched to 'gemini-2.0-flash-exp' which is free, fast, and smart.
MODEL_NAME = "models/gemini-3-flash-preview"

NAME = "Imru_Al_Qais_Poem_8"
CSV_PATH = NAME + ".csv"


In [34]:
models = client.models.list()
for m in models:
    print(m.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/embedding-001
models/text-embedding-004
models/gemini-embedding-001
models/aqa
models/

In [35]:

# ---------------------------------------------------------
# DATA LOADING
# ---------------------------------------------------------
df = pd.read_csv(CSV_PATH)

# Combine Sadr + Ajuz into one verse
df["verse"] = df["Sadr"].fillna("") + " " + df["Ajuz"].fillna("")
df["verse"] = df["verse"].str.strip()

ELEMENT_FILES = {
    "Trope": "elements/Trope.txt",
    "Similie": "elements/Similie.txt",
    "Kenning": "elements/Kenning.txt",
    "Antonymy": "elements/Antonymy.txt"
}

ELEMENT_EXPLANATIONS = {}
for name, path in ELEMENT_FILES.items():
    with open(path, "r", encoding="utf-8") as f:
        ELEMENT_EXPLANATIONS[name] = f.read().strip()


In [36]:

# ---------------------------------------------------------
# PROMPT & HELPER FUNCTIONS
# ---------------------------------------------------------
def build_prompt(verse: str, element_name: str, explanation: str) -> str:
    return f"""
Task:
Determine whether the following Arabic verse contains the rhetorical element:
**{element_name}**

Definition:
{explanation}

Rules:
- Reasoning MUST be in Arabic
- If the element does NOT appear:
    - label = 0
    - span = "NONE"
    - reasoning = "" (EMPTY STRING)
- If the element DOES appear:
    - label = 1
    - span = exact words
    - reasoning = شرح مختصر بالعربية (قصير جدًا)

Verse:
{verse}
"""

def safe_json_load(text):
    if not text or not isinstance(text, str):
        return {"label": "ERROR", "span": "ERROR", "reasoning": "Empty response"}
    
    text = re.sub(r"^```json\s*", "", text.strip())
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"label": "ERROR", "span": "ERROR", "reasoning": "JSON Decode Error"}

# Define Schema
response_schema = {
    "type": "OBJECT",
    "properties": {
        "label": {"type": "INTEGER"},
        "span": {"type": "STRING"},
        "reasoning": {"type": "STRING"}
    },
    "required": ["label", "span", "reasoning"]
}


In [37]:

# ---------------------------------------------------------
# MAIN LOOP
# ---------------------------------------------------------
for element_name, explanation in ELEMENT_EXPLANATIONS.items():
    df[f"{element_name}_label"] = None
    df[f"{element_name}_span"] = None
    df[f"{element_name}_reasoning"] = None

    print(f"\n🔍 Processing element: {element_name}")

    pbar = tqdm(total=len(df), desc=f"{element_name}", leave=True)

    for idx in range(len(df)):
        verse = df.at[idx, "verse"]
        
        if not verse:
            pbar.update(1)
            continue

        prompt = build_prompt(verse, element_name, explanation)

        # RETRY LOGIC START
        max_retries = 5
        retry_delay = 10 # Start with 10 seconds
        success = False

        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        response_schema=response_schema,
                        temperature=0.0
                    )
                )

                parsed = safe_json_load(response.text)
                
                df.at[idx, f"{element_name}_label"] = parsed.get("label")
                df.at[idx, f"{element_name}_span"] = parsed.get("span")
                df.at[idx, f"{element_name}_reasoning"] = parsed.get("reasoning")
                success = True
                break # Break retry loop on success

            except Exception as e:
                error_str = str(e)
                # Check for Rate Limit (429) errors
                if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                    wait_time = retry_delay * (attempt + 1) # Exponential backoff: 10, 20, 30...
                    print(f"\n⚠️ Quota hit. Sleeping {wait_time}s... (Attempt {attempt+1}/{max_retries})")
                    time.sleep(wait_time)
                else:
                    # If it's NOT a rate limit error (e.g., bad request), fail immediately
                    df.at[idx, f"{element_name}_label"] = "ERROR"
                    df.at[idx, f"{element_name}_reasoning"] = f"Error: {error_str}"
                    break
        
        if not success and df.at[idx, f"{element_name}_label"] is None:
             df.at[idx, f"{element_name}_label"] = "FAILED_RETRIES"

        pbar.update(1)

    pbar.close()



🔍 Processing element: Trope


Trope: 100%|██████████| 23/23 [20:35<00:00, 53.71s/it]  



🔍 Processing element: Similie


Similie: 100%|██████████| 23/23 [01:48<00:00,  4.71s/it]



🔍 Processing element: Kenning


Kenning: 100%|██████████| 23/23 [09:50<00:00, 25.69s/it] 



🔍 Processing element: Antonymy


Antonymy: 100%|██████████| 23/23 [02:06<00:00,  5.51s/it]


In [38]:

# ---------------------------------------------------------
# SAVE OUTPUT
# ---------------------------------------------------------
OUTPUT_CSV = NAME + "_analysis_gemini.csv"
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Done! Saved to {OUTPUT_CSV}")

Done! Saved to Imru_Al_Qais_Poem_8_analysis_gemini.csv
